### FEATURE ENGINEERING EXPERIMENTATION DECK

In [1]:
import pandas as pd
import numpy as np

In [26]:
df = pd.read_parquet("../data/processed/api_ingest.parquet") 
df.head()

,product_id,product_name,brand,price,product_rating,rating_count,seller_name,stock_qty,category,fetched_time,source
0,5000867,JCO4 LAPTOP BATTERY,HP,24725,0.0,0.0,Konga,1,None,2025-11-11 19:49:08+00:00,api_ingest
1,6566278,Basilisk Gaming Mouse Classic Black,Razer,244000,0.0,0.0,YOMILINCON BRAND,20,None,2025-11-11 19:49:08+00:00,api_ingest
2,5228829,Ht03xl In-built Battery,HP,35075,0.0,0.0,Konga,2,None,2025-11-11 19:49:08+00:00,api_ingest
3,2353031,HP Big-Mouth Laptop Charger - 18.5V 3.5A,HP,10000,4.5,2.0,YOMILINCON BRAND,9,None,2025-11-11 19:49:08+00:00,api_ingest
4,6861874,Baseus Ultrajoy 5w1 5-port Hub docking station...,Baseus,49000,0.0,0.0,YOMILINCON BRAND,2,None,2025-11-11 19:49:08+00:00,api_ingest


In [27]:
df.dtypes

product_id                      int64
product_name                      str
brand                             str
price                           int64
product_rating                float64
rating_count                  float64
seller_name                       str
stock_qty                       int64
category                       object
fetched_time      datetime64[us, UTC]
source                            str
dtype: object

In [28]:
df.drop(columns="category", inplace=True)
df.isna().sum()

product_id         0
product_name       0
brand              0
price              0
product_rating    40
rating_count      40
seller_name        0
stock_qty          0
fetched_time       0
source             0
dtype: int64

In [29]:
# Binning different price tiers into low, mid & high
df["price_tier"] = pd.qcut(df["price"], q=3, labels= ["Low", "Mid", "High"]) 

### BUILDING THE PRODUCT INTELLIGENCE ENGINE

In [30]:
# Checking the distribution of product_id inorder to investigate the distribution of products within the dataset
df["product_id"].value_counts()

product_id
5000867    43
6566278    22
6861874    22
6860173    22
6860155    22
           ..
6986469     1
6986465     1
6986464     1
6986463     1
6986457     1
Name: count, Length: 664, dtype: int64

In [32]:
# Sorting the dataset by fetched_time and product_id to ensure that the latest price and rating information is captured for each product 
df = df.sort_values(["fetched_time", "product_id"]) 
df.head()

,product_id,product_name,brand,price,product_rating,rating_count,seller_name,stock_qty,fetched_time,source,price_tier
3,2353031,HP Big-Mouth Laptop Charger - 18.5V 3.5A,HP,10000,4.5,2.0,YOMILINCON BRAND,9,2025-11-11 19:49:08+00:00,api_ingest,Low
0,5000867,JCO4 LAPTOP BATTERY,HP,24725,0.0,0.0,Konga,1,2025-11-11 19:49:08+00:00,api_ingest,Mid
2,5228829,Ht03xl In-built Battery,HP,35075,0.0,0.0,Konga,2,2025-11-11 19:49:08+00:00,api_ingest,Mid
39,5522604,C270 Hd Webcam - Hd 720p - Widescreen Hd Video,Logitech,80000,0.0,0.0,YOMILINCON BRAND,17,2025-11-11 19:49:08+00:00,api_ingest,Mid
38,5527693,Mini Display Port To Hdmi,Mini,36200,0.0,0.0,YOMILINCON BRAND,17,2025-11-11 19:49:08+00:00,api_ingest,Mid


In [33]:
df_2 = df.groupby("product_id", dropna=False).agg(product_name = ("product_name", "last"), brand = ("brand", "last"), seller_name = ("seller_name", "last"), source = ("source", "last"), first_seen = ("fetched_time", "first"), last_seen = ("fetched_time", "last"),
                                                   observation_count = ("product_id", "size"), avg_price = ("price", "mean"), current_price = ("price", "last"), avg_rating = ("product_rating", "mean"), current_rating = ("product_rating", "last"), 
                                                   total_rating_count = ("product_rating", "sum"), stock_qty = ("stock_qty", "last")).reset_index()
df_2.head() 

,product_id,product_name,brand,seller_name,source,first_seen,last_seen,observation_count,avg_price,current_price,avg_rating,current_rating,total_rating_count,stock_qty
0,2353031,HP Big-Mouth Laptop Charger - 18.5V 3.5A,HP,YOMILINCON BRAND,api_ingest,2025-11-11 19:49:08+00:00,2025-11-11 19:49:08+00:00,1,10000.00000,10000,4.5,4.5,4.5,9
1,4835380,Flexible Usb External Keyboard And Optical Wir...,Havit,YOMILINCON BRAND,api_ingest,2025-12-21 01:41:53+00:00,2026-01-16 01:40:10+00:00,7,10000.00000,10000,0.0,0.0,0.0,15
2,4840254,Optical Usb Mouse,Dell,KACHISON,api_ingest,2026-03-31 02:11:08+00:00,2026-03-31 02:11:08+00:00,1,4000.00000,4000,5.0,5.0,5.0,1
3,4851063,Rgb Backlit Gaming Mouse -black,Havit,YOMILINCON BRAND,api_ingest,2025-12-16 01:28:12+00:00,2025-12-26 01:27:46+00:00,3,19500.00000,19500,4.0,4.0,12.0,7
4,5000867,JCO4 Laptop Battery – High Capacity – Recharge...,HP,Konga,api_ingest,2025-11-11 19:49:08+00:00,2026-05-26 02:57:16+00:00,43,35062.44186,49420,0.0,0.0,0.0,1


### COMPLETED THE PRODUCT FEATURE ENGINEERING NEEDED FOR PRODUCT INTELLIGENCE! 

In [34]:
df_2["price_tier"] = pd.qcut(df_2["current_price"], q=3, labels= ["Low", "Mid", "High"])  

### EXPERIMENTATION COMPLETED - The brand & seller intelligence entails thesame steps which was taken in the product intelligence, Therefore, all i need to do is repurpose the steps and actions carried out in the product intelligence for the rest!! 